In [1]:
import coiled
import fsspec
import s3fs
import numpy as np
import rioxarray
import xarray as xr
import fsspec
import pandas as pd
import logging 
import numpy as np
import pytz
import dask
import re
import requests
import sparse
import time
import warnings
import zarr
from io import BytesIO
from datetime import datetime
from dask.distributed import Client, LocalCluster
from dask.distributed import print
from flox import ReindexArrayType, ReindexStrategy
from flox.xarray import xarray_reduce
from bisect import bisect_left, bisect_right

import logging
import pygwalker as pyg

# T0 INSTALL FLOX:
# 1. In home directory (cd ~), downloaded flox tar.gz (because can't install the latest version using conda-forge for some reason): wget https://files.pythonhosted.org/packages/6e/34/6eea00e3f1de745c8adad5a3dafd46c3481294cff8699c20a9b8d80502ed/flox-0.10.4.tar.gz 
# 2. Installed using pip, but it's still putting it in the active Conda environment: pip install /home/dagibbs22/flox-0.10.4.tar.gz

# TO CREATE A NOTEBOOK IN A COILED CLUSTER
# coiled notebook start --region=us-east-1

In [2]:
logging.getLogger("distributed.client").setLevel(logging.ERROR)

In [56]:
# Zarr creation cluster
cluster = coiled.Cluster(
    name="vegetation_zonal_stats",
    region="us-east-1", # close to dataset, avoid egress charges
    # n_workers=50,
    n_workers=2,
    tags={"project": "AFOLU_flux_model"},
    scheduler_vm_types="r7g.xlarge", 
    worker_vm_types="r7g.2xlarge",
    compute_purchase_option="on-demand"
)

client = cluster.get_client()

Output()

╭──────────────────────────────── Package Info ────────────────────────────────╮
│                                 ╷                                            │
│   Package                       │ Note                                       │
│ ╶───────────────────────────────┼──────────────────────────────────────────╴ │
│   coiled_local_zonal_statistics │ Source wheel built from                    │
│                                 │ /mnt/c/GIS/git/AFOLU_GHG_flux_model/src/   │
│                                 │ LULUCF/scripts/zonal_statistics            │
│   flox                          │ Wheel built from ~/flox-0.10.4.tar.gz      │
│                                 ╵                                            │
╰──────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────── Not Synced with Cluster ───────────────────────────╮
│                 ╷                                                ╷           │
│   Package       │ Error                                          │ Level     │
│ ╶───────────────┼────────────────────────────────────────────────┼─────────╴ │
│   pygwalker     │ Pip check had the following issues that need   │ Warning   │
│                 │ resolving:                                     │           │
│                 │ pygwalker 0.3.17 has requirement               │           │
│                 │ duckdb==0.9.2, but you have duckdb 1.3.0.      │           │
│                 │ pygwalker 0.3.17 has requirement               │           │
│                 │ gw-dsl-parser==0.1.8a0, but you have           │           │
│                 │ gw-dsl-parser 0.1.49.1.                        │           │
│                 │ pygwalker 0.3.17 has requirement               │           │
│                 │ segment-analytics-python==2.2.3, but you have  │           │
│                 │ segment-analytics-python 2.3.3.                │           │
│   pydantic_core │ pydantic-core~=2.33.2 has no install candidate │ Warning   │
│                 │ for Python 3.12 linux-aarch64 on conda-forge   │           │
│   lz4           │ lz4~=4.4.5 has no install candidate for Python │ Warning   │
│                 │ 3.12 linux-aarch64 on conda-forge              │           │
│   dtale         │ Pip check had the following issues that need   │ Warning   │
│                 │ resolving:                                     │           │
│                 │ dtale 3.17.0 has requirement dash<=2.18.2;     │           │
│                 │ python_version > "3.7", but you have dash      │           │
│                 │ 3.0.4.                                         │           │
│                 │ dtale 3.17.0 has requirement                   │           │
│                 │ dash-bootstrap-components<=1.7.1;              │           │
│                 │ python_version > "3.0", but you have           │           │
│                 │ dash-bootstrap-components 2.0.3.               │           │
│                 │ dtale 3.17.0 has requirement dash_daq<=0.5.0,  │           │
│                 │ but you have dash-daq 0.6.0.                   │           │
│   awscrt        │ awscrt~=0.26.1 has no install candidate for    │ Warning   │
│                 │ Python 3.12 linux-aarch64 on conda-forge       │           │
│                 ╵                                                ╵           │
╰──────────────────────────────────────────────────────────────────────────────╯

Output()

In [4]:
def timestr():

    # Define the Eastern Time timezone
    eastern = pytz.timezone('US/Eastern')

    # Get the current time in UTC and convert to Eastern Time
    eastern_time = datetime.now(eastern)

    # Format the time as a string
    return eastern_time.strftime("%Y%m%d_%H_%M_%S")

In [5]:
# Makes xarray dataframe (I think not a dataset) from list of s3 uris.
# This came from Solomon Negusse and I haven't really changed it.
# He said that an online forum suggested using xr.openmfdataset to open non-overlapping geotifs.
def make_xarray_chunks(tile_uris, chunk_size):

    xarray_chunks = xr.open_mfdataset(
        tile_uris.values.tolist(),
        parallel=True,
        chunks={'x': chunk_size, 'y':chunk_size}
    ).squeeze()

    return xarray_chunks

In [6]:
# Lists uris in an s3 folder, for creating zarr of them
# per https://chatgpt.com/g/g-vK4oPfjfp-coding-assistant/c/682201ec-1f84-800a-a9f9-c9564f613208
def list_folder_uris(base_uri):

    # Initializes S3 filesystem
    fs = s3fs.S3FileSystem(anon=False)  # Set anon=True if public bucket
    
    # Lists all files in the directory
    all_files = fs.ls(base_uri)
    
    # Filters for GeoTIFFs
    tif_files = [f"s3://{f}" for f in all_files if f.endswith(".tif")]
    
    # Converts to a Pandas Series
    series = pd.Series(tif_files)
    
    return series

# Extracts file pattern from uri. Assumes that file pattern includes _ha_yr (as it does from the LULUCF model).
def parse_pattern_from_uri(uri_series):

    uri = uri_series.values.tolist()[0]
    # print("Parsing URI:", uri)

    # regex per https://chatgpt.com/g/g-vK4oPfjfp-coding-assistant/c/681a538d-55e4-800a-818b-bcf850757ba0
    pattern = r"__([a-zA-Z0-9_]+(?:__?[a-zA-Z0-9_]+)*)_ha_yr_\d{4}.tif$"
    match = re.search(pattern, uri)

    if match:
        return match.group(1)
    else:
        return None

# Creates a Pandas dataframe with the state_nodes codes and meanings from an Excel spreadsheet
def create_state_node_df(state_node_lookup_table_local, state_node_lookup_table_s3, sheet_name):

    try:
        # Tries fetching the file from the S3 URL
        # print(f"Attempting to download file from URL: {spreadsheet}")
        response = requests.get(state_node_lookup_table_s3, timeout=10)
        response.raise_for_status()
        state_node_df = pd.read_excel(BytesIO(response.content), sheet_name=sheet_name)

    except (requests.exceptions.RequestException, Exception) as e:
        print(f"Failed to download file from S3. Falling back to local file. Error: {e}")

        print(f"Reading file from local path: {state_node_lookup_table_local}")
        state_node_df = pd.read_excel(state_node_lookup_table_local, sheet_name=sheet_name)

    return state_node_df

In [7]:
# Removes the zarr FillValue attribute from each dataset, which is necessary to avoid Float32 datatype errors
def remove_FillValue(zarr_path):

    fs = fsspec.filesystem("s3", anon=False)
    mapper = fs.get_mapper(zarr_path)
    z = zarr.open_group(mapper, mode="r+")
    
    # Loop through each array and remove _FillValue if present
    for key in z.array_keys():
        arr = z[key]
        if "_FillValue" in arr.attrs:
            print(f"   Removing _FillValue from {key}")
            del arr.attrs["_FillValue"]

    print(f"   FillValues removed from {zarr_path}")

# Crops one input to the other input's extent.
# ref is the reference dataset that is being cropped to. 
# From long chat in https://chatgpt.com/g/g-vK4oPfjfp-coding-assistant/c/684749fe-7b30-800a-ba8b-c502377f2c3a
def safe_crop(ds, ref):
    return ds.sel(x=ref.x, y=ref.y, method="nearest")

# Fix floating-point precision issues
def round_coords(ds, decimals=5):
    ds = ds.assign_coords({
        'x': np.round(ds.coords['x'].values, decimals),
        'y': np.round(ds.coords['y'].values, decimals)
    })
    return ds

In [8]:
# Converts results of flox to coordinate dictionary.
# This code came from Solomon Negusse and I haven't changed it in any substantial way.
def convert_to_coord_dict(flux_results):

    print(f"   Postprocessing: {timestr()}")
    sparse_data = flux_results.data
    
    dim_names = flux_results.dims
    indices = sparse_data.coords  # tuple of arrays with indices into each dim
    values = sparse_data.data     # non-zero values
    
    coord_dict = {
        dim: flux_results.coords[dim].values[indices[i]]
        for i, dim in enumerate(dim_names)
    }
    coord_dict["value"] = values

    return coord_dict

In [9]:
# Converts flox output to dataframe and does some processing of it:
# replaces the numeric flux type with the name
# classifies specific flux types to larger groupings
# adds the interval end year to the dataframe
# adds the state node meaning to the dataframe
# converts area from m^2 to ha
def create_df(coord_dict, state_node_df, merge_keys):

    df = pd.DataFrame(coord_dict)
    # print(df)

    # Split df into pixel area and other analysis layers
    df_area = (
        df[df['analysis_layer'] == 'pixel_area_ha']
          .rename(columns={'value': 'pixel_area_ha'})
          [merge_keys + ['pixel_area_ha']]
    )

    # Non-pixel area analysis layers
    df_other = df[df['analysis_layer'] != 'pixel_area_ha']

    # Merge area values into flux rows
    df_with_areas = df_other.merge(df_area, on=merge_keys, how='left')

    # Adds the state_node meaning and classifications to the dataframe
    df_with_areas = df_with_areas.merge(state_node_df[['state_nodes', 'meaning', 'broad_class', 'detailed_class']],
              left_on='land_state_node', right_on='state_nodes',
              how='left')
    # print("merged:", df_with_areas)

    # Replaces the year index with the actual reporting year
    df_with_areas['year'] = df_with_areas['year'] + 2016

    # Deletes redundant state node column
    df_with_areas = df_with_areas.drop(columns=['state_nodes'])

    # Converts numeric codes to ISO codes 
    # From https://github.com/wri/project-zeno-data-infra/blob/main/notebooks/grasslands_areas_gadm_2000-2022.ipynb
    if 'adm0' in df_with_areas.columns:
        df_with_areas['adm0'] = df_with_areas.adm0.apply(lambda x: numeric_to_alpha3[x])
        df_with_areas['country_name'] = df_with_areas.adm0.apply(lambda x: iso_to_country[x])
        df_with_areas['region'] = df_with_areas.adm0.apply(lambda x: iso_to_region[x])

    df_with_areas['flux_Mg_ha'] = df_with_areas['value'] / df_with_areas['pixel_area_ha'].replace(0, pd.NA)

    return df_with_areas

In [48]:
### Value options for contextual layer values.
### Every contextual layer needs to have all possible values listed here.

# Primary forest value options
primary_forest_IFL_codes = np.array([0, 1], dtype=np.uint8)

BRA_biome_codes = np.array([1, 2, 3, 4, 5, 6], dtype=np.uint8)

WDPA_codes = np.array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16], dtype=np.uint8)

# GADM v4.1 adm0 IDs (from Solomon Negusse's notebook)
gadm_adm0_ids = np.array([  0.,   4.,   8.,  10.,  12.,  16.,  20.,  24.,  28.,  31.,  32.,
        36.,  40.,  44.,  48.,  50.,  51.,  52.,  56.,  60.,  64.,  68.,
        70.,  72.,  74.,  76.,  84.,  86.,  90.,  92.,  96., 100., 104.,
       108., 112., 116., 120., 124., 132., 136., 140., 144., 148., 152.,
       156., 158., 162., 166., 170., 174., 175., 178., 180., 184., 188.,
       191., 192., 196., 203., 204., 208., 212., 214., 218., 222., 226.,
       231., 232., 233., 234., 238., 239., 242., 246., 248., 250., 254.,
       258., 260., 262., 266., 268., 270., 275., 276., 288., 292., 296.,
       300., 304., 308., 312., 316., 320., 324., 328., 332., 334., 336.,
       340., 348., 352., 356., 360., 364., 368., 372., 376., 380., 384.,
       388., 392., 398., 400., 404., 408., 410., 414., 417., 418., 422.,
       426., 428., 430., 434., 438., 440., 442., 450., 454., 458., 462.,
       466., 470., 474., 478., 480., 484., 492., 496., 498., 499., 500.,
       504., 508., 512., 516., 520., 524., 528., 531., 533., 534., 535.,
       540., 548., 554., 558., 562., 566.,70., 574., 578., 580., 581.,
       583., 584., 585., 586., 591., 598., 600., 604., 608., 612., 616.,
       620., 624., 626., 630., 634., 638., 642., 643., 646., 652., 654.,
       659., 660., 662., 663., 666., 670., 674., 678., 682., 686., 688.,
       690., 694., 702., 703., 704., 705., 706., 710., 716., 724., 728.,
       729., 732., 740., 744., 748., 752., 756., 760., 762., 764., 768.,
       772., 776., 780., 784., 788., 792., 795., 796., 798., 800., 804.,
       807., 818., 826., 831., 832., 833., 834., 840., 850., 854., 858.,
       860., 862., 876., 882., 887., 894.], dtype=np.uint16)

cont_eco_codes = np.array([0,
    1004, 1007, 1008, 1009, 1010, 1014, 1016, 1017, 1018, 1018,
    1019, 1020, 1021, 1022,
    2001, 2002, 2003, 2004, 2004, 2005, 2006, 2007, 2007, 2008,
    2008, 2009, 2009, 2010, 2010, 2011, 2012, 2013, 2013,
    2014, 2014, 2014, 2014, 2015, 2015, 2016, 2017, 2017,
    2018, 2018, 2018, 2019, 2019, 2020, 2020, 2021, 2021, 2022,
    3004, 3005,
    4001, 4002, 4003, 4004, 4005, 4006, 4008, 4009, 4011, 4012,
    4013, 4014, 4015, 4016, 4017, 4018, 4019, 4020, 4022,
    5007, 5010, 5021,
    6007, 6010, 6021,
    7001, 7002, 7003, 7004, 7005, 7007, 7009, 7011, 7012,
    7013, 7014, 7015, 7022,
    8004, 8008, 8013, 8014
], dtype=np.uint16)

In [11]:
# Converts numeric ISO values to ISO codes
# From https://github.com/wri/project-zeno-data-infra/blob/main/notebooks/grasslands_areas_gadm_2000-2022.ipynb
numeric_to_alpha3 = {
    4: 'AFG', 248: 'ALA', 8: 'ALB', 12: 'DZA', 16: 'ASM', 20: 'AND', 24: 'AGO', 660: 'AIA',
    10: 'ATA', 28: 'ATG', 32: 'ARG', 51: 'ARM', 533: 'ABW', 36: 'AUS', 40: 'AUT', 31: 'AZE',
    44: 'BHS', 48: 'BHR', 50: 'BGD', 52: 'BRB', 112: 'BLR', 56: 'BEL', 84: 'BLZ', 204: 'BEN',
    60: 'BMU', 64: 'BTN', 68: 'BOL', 535: 'BES', 70: 'BIH', 72: 'BWA', 74: 'BVT', 76: 'BRA',
    86: 'IOT', 96: 'BRN', 100: 'BGR', 854: 'BFA', 108: 'BDI', 132: 'CPV', 116: 'KHM', 120: 'CMR',
    124: 'CAN', 136: 'CYM', 140: 'CAF', 148: 'TCD', 152: 'CHL', 156: 'CHN', 162: 'CXR', 166: 'CCK',
    170: 'COL', 174: 'COM', 178: 'COG', 180: 'COD', 184: 'COK', 188: 'CRI', 384: 'CIV', 191: 'HRV',
    192: 'CUB', 531: 'CUW', 196: 'CYP', 203: 'CZE', 208: 'DNK', 262: 'DJI', 212: 'DMA', 214: 'DOM',
    218: 'ECU', 818: 'EGY', 222: 'SLV', 226: 'GNQ', 232: 'ERI', 233: 'EST', 748: 'SWZ', 231: 'ETH',
    238: 'FLK', 234: 'FRO', 242: 'FJI', 246: 'FIN', 250: 'FRA', 254: 'GUF', 258: 'PYF', 260: 'ATF',
    266: 'GAB', 270: 'GMB', 268: 'GEO', 276: 'DEU', 288: 'GHA', 292: 'GIB', 300: 'GRC', 304: 'GRL',
    308: 'GRD', 312: 'GLP', 316: 'GUM', 320: 'GTM', 831: 'GGY', 324: 'GIN', 624: 'GNB', 328: 'GUY',
    332: 'HTI', 334: 'HMD', 336: 'VAT', 340: 'HND', 344: 'HKG', 348: 'HUN', 352: 'ISL', 356: 'IND',
    360: 'IDN', 364: 'IRN', 368: 'IRQ', 372: 'IRL', 833: 'IMN', 376: 'ISR', 380: 'ITA', 388: 'JAM',
    392: 'JPN', 832: 'JEY', 400: 'JOR', 398: 'KAZ', 404: 'KEN', 296: 'KIR', 408: 'PRK', 410: 'KOR',
    414: 'KWT', 417: 'KGZ', 418: 'LAO', 428: 'LVA', 422: 'LBN', 426: 'LSO', 430: 'LBR', 434: 'LBY',
    438: 'LIE', 440: 'LTU', 442: 'LUX', 446: 'MAC', 450: 'MDG', 454: 'MWI', 458: 'MYS', 462: 'MDV',
    466: 'MLI', 470: 'MLT', 584: 'MHL', 474: 'MTQ', 478: 'MRT', 480: 'MUS', 175: 'MYT', 484: 'MEX',
    583: 'FSM', 498: 'MDA', 492: 'MCO', 496: 'MNG', 499: 'MNE', 500: 'MSR', 504: 'MAR', 508: 'MOZ',
    104: 'MMR', 516: 'NAM', 520: 'NRU', 524: 'NPL', 528: 'NLD', 540: 'NCL', 554: 'NZL', 558: 'NIC',
    562: 'NER', 566: 'NGA', 570: 'NIU', 574: 'NFK', 807: 'MKD', 580: 'MNP', 578: 'NOR', 512: 'OMN',
    586: 'PAK', 585: 'PLW', 275: 'PSE', 591: 'PAN', 598: 'PNG', 600: 'PRY', 604: 'PER', 608: 'PHL',
    612: 'PCN', 616: 'POL', 620: 'PRT', 630: 'PRI', 634: 'QAT', 638: 'REU', 642: 'ROU', 643: 'RUS',
    646: 'RWA', 652: 'BLM', 654: 'SHN', 659: 'KNA', 662: 'LCA', 663: 'MAF', 666: 'SPM', 670: 'VCT',
    882: 'WSM', 674: 'SMR', 678: 'STP', 682: 'SAU', 686: 'SEN', 688: 'SRB', 690: 'SYC', 694: 'SLE',
    702: 'SGP', 534: 'SXM', 703: 'SVK', 705: 'SVN', 90: 'SLB', 706: 'SOM', 710: 'ZAF', 239: 'SGS',
    728: 'SSD', 724: 'ESP', 144: 'LKA', 729: 'SDN', 740: 'SUR', 744: 'SJM', 752: 'SWE', 756: 'CHE',
    760: 'SYR', 158: 'TWN', 762: 'TJK', 834: 'TZA', 764: 'THA', 626: 'TLS', 768: 'TGO', 772: 'TKL',
    776: 'TON', 780: 'TTO', 788: 'TUN', 792: 'TUR', 795: 'TKM', 796: 'TCA', 798: 'TUV', 800: 'UGA',
    804: 'UKR', 784: 'ARE', 826: 'GBR', 840: 'USA', 581: 'UMI', 858: 'URY', 860: 'UZB', 548: 'VUT',
    862: 'VEN', 704: 'VNM', 92: 'VGB', 850: 'VIR', 876: 'WLF', 732: 'ESH', 887: 'YEM', 894: 'ZMB',
    716: 'ZWE', 0: 'NA'
}

iso_to_country = {
    'ABW': 'Aruba', 'AFG': 'Afghanistan', 'AGO': 'Angola', 'AIA': 'Anguilla', 'ALA': 'Åland Islands', 'ALB': 'Albania', 'AND': 'Andorra', 'ARE': 'United Arab Emirates', 'ARG': 'Argentina',
    'ARM': 'Armenia', 'ATF': 'French Southern Territories', 'ATG': 'Antigua and Barbuda', 'AUS': 'Australia', 'AUT': 'Austria', 'AZE': 'Azerbaijan', 'BDI': 'Burundi', 'BEL': 'Belgium', 'BEN': 'Benin',
    'BES': 'Bonaire', 'BFA': 'Burkina Faso', 'BGD': 'Bangladesh', 'BGR': 'Bulgaria', 'BHR': 'Bahrain', 'BHS': 'Bahamas', 'BIH': 'Bosnia and Herzegovina', 'BLM': 'Saint Barthélemy', 'BLR': 'Belarus',
    'BLZ': 'Belize', 'BMU': 'Bermuda', 'BOL': 'Bolivia', 'BRA': 'Brazil', 'BRB': 'Barbados', 'BRN': 'Brunei', 'BTN': 'Bhutan', 'BWA': 'Botswana', 'CAF': 'Central African Republic', 'CAN': 'Canada',
    'CHE': 'Switzerland', 'CHL': 'Chile', 'CHN': 'China', 'CIV': 'Côte d Ivoire', 'CMR': 'Cameroon', 'COD': 'Democratic Republic of the Congo', 'COG': 'Republic of Congo', 'COL': 'Colombia',
    'COM': 'Comoros', 'CPV': 'Cape Verde', 'CRI': 'Costa Rica', 'CUB': 'Cuba', 'CUW': 'Curaçao', 'CYM': 'Cayman Islands', 'CYP': 'Cyprus', 'CZE': 'Czechia', 'DEU': 'Germany', 'DJI': 'Djibouti',
    'DMA': 'Dominica', 'DNK': 'Denmark', 'DOM': 'Dominican Republic', 'DZA': 'Algeria', 'ECU': 'Ecuador', 'EGY': 'Egypt', 'ERI': 'Eritrea', 'ESH': 'Western Sahara', 'ESP': 'Spain', 'EST': 'Estonia',
    'ETH': 'Ethiopia', 'FIN': 'Finland', 'FJI': 'Fiji', 'FLK': 'Falkland Islands', 'FRA': 'France', 'FRO': 'Faroe Islands', 'FSM': 'Micronesia (Federated States of)', 'GAB': 'Gabon', 'GBR': 'United Kingdom of Great Britain and Northern Ireland',
    'GEO': 'Georgia', 'GGY': 'Guernsey', 'GHA': 'Ghana', 'GIB': 'Gibraltar', 'GIN': 'Guinea', 'GLP': 'Guadeloupe', 'GMB': 'Gambia', 'GNB': 'Guinea-Bissau', 'GNQ': 'Equatorial Guinea', 'GRC': 'Greece',
    'GRD': 'Grenada', 'GRL': 'Greenland', 'GTM': 'Guatemala', 'GUF': 'French Guiana', 'GUY': 'Guyana', 'HKG': 'Hong Kong', 'HND': 'Honduras', 'HRV': 'Croatia', 'HTI': 'Haiti', 'HUN': 'Hungary',
    'IDN': 'Indonesia', 'IMN': 'Isle of Man', 'IND': 'India', 'IRL': 'Ireland', 'IRN': 'Iran', 'IRQ': 'Iraq', 'ISL': 'Iceland', 'ISR': 'Israel', 'ITA': 'Italy', 'JAM': 'Jamaica', 'JEY': 'Jersey',
    'JOR': 'Jordan', 'JPN': 'Japan', 'KAZ': 'Kazakhstan', 'KEN': 'Kenya', 'KGZ': 'Kyrgyzstan', 'KHM': 'Cambodia', 'KIR': 'Kiribati', 'KNA': 'Saint Kitts and Nevis', 'KOR': 'Korea', 'KWT': 'Kuwait',
    'LAO': 'Laos', 'LBN': 'Lebanon', 'LBR': 'Liberia', 'LBY': 'Libya', 'LCA': 'Saint Lucia', 'LIE': 'Liechtenstein', 'LKA': 'Sri Lanka', 'LSO': 'Lesotho', 'LTU': 'Lithuania', 'LUX': 'Luxembourg', 
    'LVA': 'Latvia', 'MAC': 'Macao', 'MAF': 'Saint Martin (French part)', 'MAR': 'Morocco', 'MCO': 'Monaco', 'MDA': 'Moldova', 'MDG': 'Madagascar', 'MDV': 'Maldives',
    'MEX': 'Mexico', 'MKD': 'North Macedonia', 'MLI': 'Mali', 'MLT': 'Malta', 'MMR': 'Myanmar', 'MNE': 'Montenegro', 'MNG': 'Mongolia', 'MOZ': 'Mozambique', 'MRT': 'Mauritania', 'MSR': 'Montserrat', 'MTQ': 'Martinique', 'MUS': 'Mauritius',
    'MWI': 'Malawi',
    'MYS': 'Malaysia',
    'MYT': 'Mayotte',
    'NAM': 'Namibia',
    'NCL': 'New Caledonia',
    'NER': 'Niger',
    'NFK': 'Norfolk Island',
    'NGA': 'Nigeria',
    'NIC': 'Nicaragua',
    'NLD': 'Netherlands',
    'NOR': 'Norway',
    'NPL': 'Nepal',
    'NRU': 'Nauru',
    'NZL': 'New Zealand',
    'OMN': 'Oman',
    'PAK': 'Pakistan',
    'PAN': 'Panama',
    'PER': 'Peru',
    'PHL': 'Philippines',
    'PLW': 'Palau',
    'PNG': 'Papua New Guinea',
    'POL': 'Poland',
    'PRI': 'Puerto Rico',
    'PRK': 'Korea (the Democratic Peoples Republic of)',
    'PRT': 'Portugal',
    'PRY': 'Paraguay',
    'PSE': 'Palestine, State of',
    'QAT': 'Qatar',
    'REU': 'Réunion',
    'ROU': 'Romania',
    'RUS': 'Russian Federation',
    'RWA': 'Rwanda',
    'SAU': 'Saudi Arabia',
    'SDN': 'Sudan',
    'SEN': 'Senegal',
    'SGP': 'Singapore',
    'SJM': 'Svalbard and Jan Mayen',
    'SLB': 'Solomon Islands',
    'SLE': 'Sierra Leone',
    'SLV': 'El Salvador',
    'SMR': 'San Marino',
    'SOM': 'Somalia',
    'SPM': 'Saint Pierre and Miquelon',
    'SRB': 'Serbia',
    'SSD': 'South Sudan',
    'STP': 'Sao Tome and Principe',
    'SUR': 'Suriname',
    'SVK': 'Slovakia',
    'SVN': 'Slovenia',
    'SWE': 'Sweden',
    'SWZ': 'Swaziland',
    'SXM': 'Sint Maarten',
    'SYC': 'Seychelles',
    'SYR': 'Syria',
    'TCA': 'Turks and Caicos Islands',
    'TCD': 'Chad',
    'TGO': 'Togo',
    'THA': 'Thailand',
    'TJK': 'Tajikistan',
    'TKM': 'Turkmenistan',
    'TLS': 'East Timor',
    'TTO': 'Trinidad and Tobago',
    'TUN': 'Tunisia',
    'TUR': 'Turkey',
    'TUV': 'Tuvalu',
    'TWN': 'Taiwan',
    'TZA': 'Tanzania',
    'UGA': 'Uganda',
    'UKR': 'Ukraine',
    'UMI': 'United States Minor Outlying Islands',
    'URY': 'Uruguay',
    'USA': 'United States of America (the)',
    'UZB': 'Uzbekistan',
    'VAT': 'Holy See',
    'VCT': 'Saint Vincent and the Grenadines',
    'VEN': 'Venezuela',
    'VGB': 'British Virgin Islands',
    'VIR': 'Virgin Islands, U.S.',
    'VNM': 'Vietnam',
    'VUT': 'Vanuatu',
    'XAD': 'nan',
    'XCA': 'nan',
    'XCL': 'nan',
    'XKO': 'nan',
    'XNC': 'nan',
    'XPI': 'nan',
    'XSP': 'nan',
    'YEM': 'Yemen',
    'ZAF': 'South Africa',
    'ZMB': 'Zambia',
    'ZWE': 'Zimbabwe'
}

iso_to_region = {
    'ABW': 'Tropical LAC', 'AFG': 'Non-tropical Asia', 'AGO': 'Tropical Africa', 'AIA': 'Tropical LAC', 'ALA': 'Europe', 'ALB': 'Europe', 'AND': 'Europe', 'ARE': 'Non-tropical Asia',
    'ARG': 'Non-tropical LAC', 'ARM': 'Non-tropical Asia', 'ATF': 'Non-tropical Africa', 'ATG': 'Tropical LAC', 'AUS': 'Non-tropical Asia', 'AUT': 'Europe', 'AZE': 'Non-tropical Asia',
    'BDI': 'Tropical Africa', 'BEL': 'Europe', 'BEN': 'Tropical Africa', 'BES': 'Tropical LAC', 'BFA': 'Tropical Africa', 'BGD': 'Tropical Asia', 'BGR': 'Europe', 'BHR': 'Non-tropical Asia',
    'BHS': 'Tropical LAC', 'BIH': 'Europe', 'BLM': 'Tropical LAC', 'BLR': 'Europe', 'BLZ': 'Tropical LAC', 'BMU': 'Tropical LAC', 'BOL': 'Tropical LAC', 'BRA': 'Tropical LAC', 'BRB': 'Tropical LAC', 
    'BRN': 'Tropical Asia', 'BTN': 'Tropical Asia', 'BWA': 'Tropical Africa', 'CAF': 'Tropical Africa', 'CAN': 'North America', 'CHE': 'Europe', 'CHL': 'Non-tropical LAC', 'CHN': 'Non-tropical Asia',
    'CIV': 'Tropical Africa', 'CMR': 'Tropical Africa', 'COD': 'Tropical Africa', 'COG': 'Tropical Africa', 'COL': 'Tropical LAC', 'COM': 'Tropical Africa', 'CPV': 'Tropical Africa', 
    'CRI': 'Tropical LAC', 'CUB': 'Tropical LAC', 'CUW': 'Tropical LAC', 'CYM': 'Tropical LAC', 'CYP': 'Europe', 'CZE': 'Europe', 'DEU': 'Europe', 'DJI': 'Tropical Africa', 'DMA': 'Tropical LAC',
    'DNK': 'Europe', 'DOM': 'Tropical LAC', 'DZA': 'Non-tropical Africa', 'ECU': 'Tropical LAC', 'EGY': 'Non-tropical Africa', 'ERI': 'Tropical Africa', 'ESH': 'Non-tropical Africa', 'ESP': 'Europe', 'EST': 'Europe',
    'ETH': 'Tropical Africa', 'FIN': 'Europe', 'FJI': 'Tropical Asia', 'FLK': 'Non-tropical LAC', 'FRA': 'Europe', 'FRO': 'Europe', 'FSM': 'Tropical Asia', 'GAB': 'Tropical Africa', 'GBR': 'Europe',
    'GEO': 'Non-tropical Asia', 'GGY': 'Europe', 'GHA': 'Tropical Africa', 'GIB': 'Europe', 'GIN': 'Tropical Africa', 'GLP': 'Tropical LAC', 'GMB': 'Tropical Africa', 'GNB': 'Tropical Africa',
    'GNQ': 'Tropical Africa', 'GRC': 'Europe', 'GRD': 'Tropical LAC', 'GRL': 'Europe', 'GTM': 'Tropical LAC', 'GUF': 'Tropical LAC', 'GUY': 'Tropical LAC', 'HKG': 'Non-tropical Asia', 'HND': 'Tropical LAC',
    'HRV': 'Europe', 'HTI': 'Tropical LAC', 'HUN': 'Europe', 'IDN': 'Tropical Asia', 'IMN': 'Europe', 'IND': 'Tropical Asia', 'IRL': 'Europe', 'IRN': 'Non-tropical Asia', 'IRQ': 'Non-tropical Asia',
    'ISL': 'Europe', 'ISR': 'Non-tropical Asia', 'ITA': 'Europe', 'JAM': 'Tropical LAC', 'JEY': 'Europe', 'JOR': 'Non-tropical Asia', 'JPN': 'Non-tropical Asia', 'KAZ': 'Non-tropical Asia', 'KEN': 'Tropical Africa',
    'KGZ': 'Non-tropical Asia', 'KHM': 'Tropical Asia', 'KIR': 'Tropical Asia', 'KNA': 'Tropical LAC', 'KOR': 'Non-tropical Asia', 'KWT': 'Non-tropical Asia', 'LAO': 'Tropical Asia', 
    'LBN': 'Non-tropical Asia', 'LBR': 'Tropical Africa', 'LBY': 'Non-tropical Africa', 'LCA': 'Tropical LAC', 'LIE': 'Europe', 'LKA': 'Tropical Asia', 'LSO': 'Non-tropical Africa', 'LTU': 'Europe',
    'LUX': 'Europe', 'LVA': 'Europe', 'MAC': 'Tropical Asia', 'MAF': 'Tropical LAC', 'MAR': 'Non-tropical Africa', 'MCO': 'Europe', 'MDA': 'Europe', 'MDG': 'Tropical Africa', 'MDV': 'Tropical Africa',
    'MEX': 'Tropical LAC', 'MKD': 'Europe', 'MLI': 'Tropical Africa', 'MLT': 'Europe', 'MMR': 'Tropical Asia', 'MNE': 'Europe', 'MNG': 'Non-tropical Asia', 'MOZ': 'Tropical Africa', 'MRT': 'Tropical Africa',
    'MSR': 'Tropical LAC', 'MTQ': 'Tropical LAC', 'MUS': 'Tropical Africa', 'MWI': 'Tropical Africa', 'MYS': 'Tropical Asia', 'MYT': 'Tropical Africa', 'NAM': 'Tropical Africa', 'NCL': 'Tropical Asia', 'NER': 'Tropical Africa',
    'NFK': 'Non-tropical Asia', 'NGA': 'Tropical Africa', 'NIC': 'Tropical LAC', 'NLD': 'Europe', 'NOR': 'Europe', 'NPL': 'Tropical Asia', 'NRU': 'Non-tropical Asia', 'NZL': 'Non-tropical Asia',
    'OMN': 'Non-tropical Asia', 'PAK': 'Non-tropical Asia', 'PAN': 'Tropical LAC', 'PER': 'Tropical LAC', 'PHL': 'Tropical Asia', 'PLW': 'Tropical Asia', 'PNG': 'Tropical Asia', 'POL': 'Europe',
    'PRI': 'Tropical LAC', 'PRK': 'Non-tropical Asia', 'PRT': 'Europe', 'PRY': 'Tropical LAC', 'PSE': 'Non-tropical Asia', 'QAT': 'Non-tropical Asia', 'REU': 'Tropical Africa', 'ROU': 'Europe',
    'RUS': 'Non-tropical Asia', 'RWA': 'Tropical Africa', 'SAU': 'Non-tropical Asia', 'SDN': 'Tropical Africa', 'SEN': 'Tropical Africa', 'SGP': 'Tropical Asia', 'SJM': 'Europe', 'SLB': 'Tropical Asia',
    'SLE': 'Tropical Africa', 'SLV': 'Tropical LAC', 'SMR': 'Europe', 'SOM': 'Tropical Africa', 'SPM': 'North America', 'SRB': 'Europe', 'SSD': 'Tropical Africa', 'STP': 'Non-tropical Africa', 'SUR': 'Tropical LAC',
    'SVK': 'Europe', 'SVN': 'Europe', 'SWE': 'Europe', 'SWZ': 'Tropical Africa', 'SXM': 'Tropical LAC', 'SYC': 'Tropical Africa', 'SYR': 'Non-tropical Asia', 'TCA': 'Tropical LAC', 'TCD': 'Tropical Africa',
    'TGO': 'Tropical Africa', 'THA': 'Tropical Asia', 'TJK': 'Non-tropical Asia', 'TKM': 'Non-tropical Asia', 'TLS': 'Tropical Asia', 'TTO': 'Tropical LAC', 'TUN': 'Non-tropical Africa',
    'TUR': 'Non-tropical Asia', 'TUV': 'Tropical Asia', 'TWN': 'Non-tropical Asia', 'TZA': 'Tropical Africa', 'UGA': 'Tropical Africa', 'UKR': 'Europe', 'UMI': 'Tropical Asia', 'URY': 'Non-tropical LAC',
    'USA': 'North America', 'UZB': 'Non-tropical Asia', 'VAT': 'Europe', 'VCT': 'Tropical LAC', 'VEN': 'Tropical LAC', 'VGB': 'Tropical LAC', 'VIR': 'Tropical LAC', 'VNM': 'Tropical Asia',
    'VUT': 'Tropical Asia', 'XAD': 'Not tropical misc', 'XCA': 'Not tropical misc', 'XCL': 'Not tropical misc', 'XKO': 'Not tropical misc', 'XNC': 'Not tropical misc', 'XPI': 'Not tropical misc',
    'XSP': 'Not tropical misc', 'YEM': 'Non-tropical Asia', 'ZAF': 'Non-tropical Africa', 'ZMB': 'Tropical Africa', 'ZWE': 'Tropical Africa'
}

Code to run zonal stats

In [12]:
# General zonal stat run properties

model_version = "version_1_0_5__standard__global"  # model version, from s3 paths that are being read
run_date = "20260130"   # model run date, from s3 paths that are being read
chunk_size = 4000  # pixels

interval_label = '2016'

# s3 folders for model outputs being analyzed
output_path = f"s3://gfw2-data/climate/AFOLU_flux_model/LULUCF/outputs_vegetation/{model_version}/"  # Model output path, for inputs to zonal stats

# Analysis layer s3 paths
gross_emis_CO2_folder = f"{output_path}gross_emissions__all_C_pools__CO2_only__MgCO2/standard_model/annual_intervals/INTERVAL/_ha_yr/4000_pixels/{run_date}/"
gross_emis_non_CO2_folder = f"{output_path}gross_emissions__all_C_pools__non_CO2_only__MgCO2e/standard_model/annual_intervals/INTERVAL/_ha_yr/4000_pixels/{run_date}/"
gross_emis_all_gases_folder = f"{output_path}gross_emissions__all_C_pools__all_gases__MgCO2e/standard_model/annual_intervals/INTERVAL/_ha_yr/4000_pixels/{run_date}/"
gross_remv_all_pools_folder = f"{output_path}gross_removals__all_C_pools__MgCO2/standard_model/annual_intervals/INTERVAL/_ha_yr/4000_pixels/{run_date}/"
net_flux_all_pools_CO2_folder = f"{output_path}net_flux__all_C_pools__CO2_only__MgCO2/standard_model/annual_intervals/INTERVAL/_ha_yr/4000_pixels/{run_date}/"
net_flux_all_pools_all_gases_folder = f"{output_path}net_flux__all_C_pools__all_gases__MgCO2e/standard_model/annual_intervals/INTERVAL/_ha_yr/4000_pixels/{run_date}/"
node_folder = f"{output_path}land_state_node/standard_model/annual_intervals/INTERVAL/4000_pixels/{run_date}/"

# Vegetation mega-zarr
veg_mega_zarr_s3_path = f"{output_path}mega_zarr/annual_intervals/{chunk_size}_pixels/{run_date}/vegetation_zarr.zarr"

# zarrs for layers not from the flux model (only need to created once)
# They are in a central folder, not with their specific geotif tile sets (at least for now-- we could change this)
adm0_folder = "s3://gfw2-data/gadm_administrative_boundaries/v4.1/v4.1.64__from_gfw-data-lake/raster/epsg-4326/10/40000/adm0/gdal-geotiff/" #GADM v4.1
adm0_zarr_path = "s3://gfw2-data/climate/AFOLU_flux_model/contextual_layer_global_zarr/GADM4_1_adm0_global/20251209_fillValue_removed/global_GADM41_adm0_20251209.zarr"

pixel_area_folder = "s3://gfw2-data/analyses/umd_area_2013__from_gfw-data-lake/v1.10/raster/epsg-4326/10/40000/area_m/gdal-geotiff/"
pixel_area_zarr_path = "s3://gfw2-data/climate/AFOLU_flux_model/contextual_layer_global_zarr/pixel_area/20251209_fillValue_removed/global_pixel_area_20251209.zarr"

primary_forest_IFL_folder = "s3://gfw2-data/climate/carbon_model/ifl_primary_merged/processed/20200724/"
primary_forest_IFL_zarr_path = "s3://gfw2-data/climate/AFOLU_flux_model/contextual_layer_global_zarr/IFL2000_tropical_primary_forest_2001/20251209_fillValue_removed/ifl_primary_forest_merged_20251209.zarr"

wdpa_folder = "s3://gfw2-data/conservation/wdpa_licensed_proteced_areas__from_data_lake/v202511/raster/epsg-4326/10/40000/detailed_iucn_cat/gdal-geotiff/"
wdpa_zarr_path = "s3://gfw2-data/climate/AFOLU_flux_model/contextual_layer_global_zarr/WDPAv202511/20251229_fillValue_removed/wdpa_20251229.zarr"

BRA_biomes_folder = "s3://gfw2-data/country/bra/bra_biomes_geotif/"
BRA_biomes_zarr_path = "s3://gfw2-data/climate/AFOLU_flux_model/contextual_layer_global_zarr/BRA_biomes/20251229_fillValue_removed/BRA_biomes_20251229.zarr"

cont_eco_folder = "s3://gfw2-data/climate/carbon_model/fao_ecozones/ecozone_continent/20190116/processed/"
cont_eco_zarr_path = "s3://gfw2-data/climate/AFOLU_flux_model/contextual_layer_global_zarr/FAO_ecozone_continents/20260206_fillValue_removed/FAO_ecozone_continents_20260206.zarr"


# Spreadsheet for state_node meanings (local computer and s3 locations)
state_node_lookup_table_local = "/mnt/c/GIS/git/AFOLU_GHG_flux_model/src/LULUCF/LULUCF_state_node_lookup_table.xlsx"
state_node_lookup_table_s3 = "http://gfw2-data.s3.amazonaws.com/climate/AFOLU_flux_model/LULUCF/state_node_lookup_tables/LULUCF_state_node_lookup_table.xlsx"
sheet = "v102_20251027"

In [ ]:
%%time

# # # CREATES ZARRS FOR INPUTS NOT GENERATED BY THE AFOLU MODEL
# # # THIS SHOULD ONLY EVER HAVE TO BE DONE ONCE FOR EACH INPUT
# # # Creating adm0, pixel area and IFL/primary forest used 54 credits $3.05 AWS charges, and 7 minutes (50 r7g.2xlarge workers).
# # # https://cloud.coiled.io/clusters/1311444/account/wri-forest-research/information?workspace=WRI-forest-research

# # print(f"Reading inputs that apply to all intervals: {timestr()}")

# # GADM adm0
# adm0_uris = list_folder_uris(adm0_folder)
# print("adm0_folder:", adm0_folder)
# print(adm0_uris[0])
# print(f"Tile count in {adm0_folder}: {len(adm0_uris)}")

# print(f"   Reading adm0: {timestr()}")
# adm0_xarray_chunks = make_xarray_chunks(adm0_uris, chunk_size)
# adm0_xarray_chunks['band_data'] = adm0_xarray_chunks['band_data'].astype('uint16')  # adm0 should be uint16 but make_xarray_chunks makes it float64 for some reason
# # print("adm0_xarray_chunks:", adm0_xarray_chunks)  # Print to confirm that the zarr datatype is correct

# print(f"   zarring adm0: {timestr()}")
# adm0_xarray_chunks.to_zarr(adm0_zarr_path, mode='w')
# remove_FillValue(adm0_zarr_path)  # Added per https://chatgpt.com/g/g-vK4oPfjfp-coding-assistant/c/6912af84-deb4-832d-81f0-da2b22b0737d to deal with FillValue problems
# print(f"   Finished zarring adm0: {timestr()}")


# # Pixel area
# pixel_area_uris = list_folder_uris(pixel_area_folder)
# print("pixel_area_folder:", pixel_area_folder)
# print(pixel_area_uris[0])
# print(f"Tile count in {pixel_area_folder}: {len(pixel_area_uris)}")

# print(f"   Reading pixel_area: {timestr()}")
# pixel_area_xarray_chunks = make_xarray_chunks(pixel_area_uris, chunk_size)
# print("pixel_area_xarray_chunks:", pixel_area_xarray_chunks)

# print(f"   zarring pixel_area: {timestr()}")
# pixel_area_xarray_chunks.to_zarr(pixel_area_zarr_path, mode='w')
# remove_FillValue(pixel_area_zarr_path)  # Added per https://chatgpt.com/g/g-vK4oPfjfp-coding-assistant/c/6912af84-deb4-832d-81f0-da2b22b0737d to deal with FillValue problems
# print(f"   Finished zarring pixel_area: {timestr()}")


# # Humid tropical primary forest/IFL merged
# # Took 5.5 minutes with 50 workers. Got several PerformanceWarning about increasing number of chunks by factor of 10.
# # Also, Dask graph of 210MB, then a pause of several minutes. 
# primary_forest_IFL_uris = list_folder_uris(primary_forest_IFL_folder)
# print("primary_forest_IFL_folder:", primary_forest_IFL_folder)
# print(primary_forest_IFL_uris[0])
# print(f"Tile count in {primary_forest_IFL_folder}: {len(primary_forest_IFL_uris)}")

# print(f"   Reading primary_forest_IFL: {timestr()}")
# primary_forest_IFL_xarray_chunks = make_xarray_chunks(primary_forest_IFL_uris, chunk_size)
# primary_forest_IFL_xarray_chunks['band_data'] = primary_forest_IFL_xarray_chunks['band_data'].astype('uint8')  # should be uint8 but make_xarray_chunks makes it float32 for some reason
# print("primary_forest_IFL_xarray_chunks:", primary_forest_IFL_xarray_chunks)  # Print to confirm that the zarr datatype is correct

# print(f"   zarring primary_forest_IFL: {timestr()}")
# primary_forest_IFL_xarray_chunks.to_zarr(primary_forest_IFL_zarr_path, mode='w')
# remove_FillValue(primary_forest_IFL_zarr_path)  # Added per https://chatgpt.com/g/g-vK4oPfjfp-coding-assistant/c/6912af84-deb4-832d-81f0-da2b22b0737d to deal with FillValue problems
# print(f"   Finished zarring primary_forest_IFL: {timestr()}")


# # WDPA (v202511)
# # Took 8 minutes with 50 workers. Got several PerformanceWarning about increasing number of chunks by factor of 10.
# # Also, Dask graph of 306MB, then a pause of several minutes. 
# # https://cloud.coiled.io/clusters/1348054/account/wri-forest-research/information?workspace=WRI-forest-research
# wdpa_uris = list_folder_uris(wdpa_folder)
# print("wdpa_folder:", wdpa_folder)
# print(wdpa_uris[0])
# print(f"Tile count in {wdpa_folder}: {len(wdpa_uris)}")

# print(f"   Reading WDPA: {timestr()}")
# wdpa_xarray_chunks = make_xarray_chunks(wdpa_uris, chunk_size)
# wdpa_xarray_chunks['band_data'] = wdpa_xarray_chunks['band_data'].astype('uint8')  # should be uint8 but make_xarray_chunks makes it float32 for some reason
# print("wdpa_xarray_chunks:", wdpa_xarray_chunks)  # Print to confirm that the zarr datatype is correct

# print(f"   zarring WDPA: {timestr()}")
# wdpa_xarray_chunks.to_zarr(wdpa_zarr_path, mode='w')
# remove_FillValue(wdpa_zarr_path)  # Added per https://chatgpt.com/g/g-vK4oPfjfp-coding-assistant/c/6912af84-deb4-832d-81f0-da2b22b0737d to deal with FillValue problems
# print(f"   Finished zarring WDPA: {timestr()}")


# # Brazil biomes
# # Took 15 seconds with 50 workers. No warnings.
# # https://cloud.coiled.io/clusters/1348054/account/wri-forest-research/information?workspace=WRI-forest-research
# BRA_biomes_uris = list_folder_uris(BRA_biomes_folder)
# print("BRA_biomes_folder:", BRA_biomes_folder)
# print(BRA_biomes_uris[0])
# print(f"Tile count in {BRA_biomes_folder}: {len(BRA_biomes_uris)}")

# print(f"   Reading BRA biomes: {timestr()}")
# BRA_biomes_chunks = make_xarray_chunks(BRA_biomes_uris, chunk_size)
# BRA_biomes_chunks['band_data'] = BRA_biomes_chunks['band_data'].astype('uint8')  # should be uint8 but make_xarray_chunks makes it float32 for some reason
# print("BRA_biomes_chunks:", BRA_biomes_chunks)  # Print to confirm that the zarr datatype is correct

# print(f"   zarring BRA biomes: {timestr()}")
# BRA_biomes_chunks.to_zarr(BRA_biomes_zarr_path, mode='w')
# remove_FillValue(BRA_biomes_zarr_path)  # Added per https://chatgpt.com/g/g-vK4oPfjfp-coding-assistant/c/6912af84-deb4-832d-81f0-da2b22b0737d to deal with FillValue problems
# print(f"   Finished zarring BRA biomes: {timestr()}")


# # FAO ecozones 2000 x continent
# # Took 5.5 minutes with 50 workers. Several warnings of PerformanceWarning: Increasing number of chunks by factor of 10. Don't know why, and didn't try to fix.  
# # Also, Dask graph of 246MB, then a pause of several minutes. 
# # Cluster: https://cloud.coiled.io/clusters/1423362/account/wri-forest-research/information?workspace=WRI-forest-research
# cont_eco_uris = list_folder_uris(cont_eco_folder)
# print("cont_eco_folder:", cont_eco_folder)
# print(cont_eco_uris[0])
# print(f"Tile count in {cont_eco_folder}: {len(cont_eco_uris)}")

# print(f"   Reading continent x ecozone: {timestr()}")
# cont_eco_chunks = make_xarray_chunks(cont_eco_uris, chunk_size)
# cont_eco_chunks['band_data'] = cont_eco_chunks['band_data'].astype('uint16')  # should be uint16 but make_xarray_chunks makes it float32 for some reason
# print("cont_eco_chunks:", cont_eco_chunks)  # Print to confirm that the zarr datatype is correct

# print(f"   zarring cont_eco: {timestr()}")
# cont_eco_chunks.to_zarr(cont_eco_zarr_path, mode='w')
# remove_FillValue(cont_eco_zarr_path)  # Added per https://chatgpt.com/g/g-vK4oPfjfp-coding-assistant/c/6912af84-deb4-832d-81f0-da2b22b0737d to deal with FillValue problems
# print(f"   Finished zarring cont_eco: {timestr()}")

In [34]:
# # Checks contextual layers (no temporal dimension; accounts for zarr not necessarily being global and so the indices are not global)
# # Per https://chatgpt.com/g/g-p-69399a7fcc808191b337d3fac695447c-afolu-flux-model/c/6986043f-c8b0-832c-837f-7329873aa948

# bounds = [13, 48, 14, 49]  # For GADM adm0: Three countries meet in Europe, with different values in three corners (50N_010E)
# zarr_path = adm0_zarr_path  # works

# # bounds = [119, -3, 120, -2]  # For IFL/primary forest: Extensive primary forest, should have primary forest (1) in lower-left and upper-right corners (00N_110E)
# # zarr_path = primary_forest_IFL_zarr_path   # works

# # bounds = [13, 48, 14, 49]  # For pixel area (50N_010E)
# # zarr_path = pixel_area_zarr_path  # works

# # bounds = [-58, -16, -57, -15]  # For Brazil biomes: Three biomes meet, with different values in three corners (10S_060W)
# # zarr_path = BRA_biomes_zarr_path  # works

# # bounds = [21, -3, 22, -2]  # For WDPA: has WDPA 0, 3 (bottom left, top right), and 9 (top left) (00N_020E)
# # zarr_path = wdpa_zarr_path  # works

# # bounds = [119, -6, 120, -5]  # For continent-ecozone: mix of 0, 4018 and 4020, with 4020 in upper right (00N_110E)
# # zarr_path = cont_eco_zarr_path  # works

# def get_index_range(coords, min_val, max_val, descending=False):
#     if descending:
#         coords = coords[::-1]
#         i0 = bisect_left(coords, max_val)
#         i1 = bisect_right(coords, min_val)
#         return len(coords) - i1, len(coords) - i0
#     else:
#         i0 = bisect_left(coords, min_val)
#         i1 = bisect_right(coords, max_val)
#         return i0, i1

# print(f"Getting stats for {bounds}")
# fs = fsspec.filesystem("s3", anon=False)

# # zarrs without time dimension have the variable name band_data
# var_name = 'band_data'

# # print(f"Getting array for {bounds_str}")
# zarr_mapper_band_data = fs.get_mapper(f"{zarr_path}/{var_name}")
# zarr_array = zarr.open_array(zarr_mapper_band_data, mode="r")

# # Step 1: Points to the Zarr group
# fs = fsspec.filesystem("s3", anon=False)
# zarr_store = fs.get_mapper(zarr_path)
# zarr_group = zarr.open_group(zarr_store, mode="r")

# # Step 2: Lists keys
# print("Zarr keys:", list(zarr_group.array_keys()))

# # Step 3: Tries reading coordinate arrays
# if "y" in zarr_group:
#     y_coords = zarr_group["y"][:]
#     print("y shape:", y_coords.shape)
# else:
#     sys.exit("No y coordinate array")

# if "x" in zarr_group:
#     x_coords = zarr_group["x"][:]
#     print("x shape:", x_coords.shape)
# else:
#     sys.exit("No x coordinate array")

# lat_vals = y_coords  # usually descending
# lon_vals = x_coords  # usually ascending

# lat0, lat1 = get_index_range(lat_vals, bounds[3], bounds[1], descending=True)
# lon0, lon1 = get_index_range(lon_vals, bounds[0], bounds[2])

# zarr_chunk_array = zarr_array[lat0:lat1, lon0:lon1]
# print("zarr_chunk_array:", zarr_chunk_array)

# print("min:", float(np.nanmin(zarr_chunk_array)))
# print("mean:", float(np.nanmean(zarr_chunk_array)))
# print("max:", float(np.nanmax(zarr_chunk_array)))
# print("count:", np.count_nonzero(zarr_chunk_array))

Getting stats for [13, 48, 14, 49]
Zarr keys: ['band', 'band_data', 'spatial_ref', 'x', 'y']
y shape: (720000,)
x shape: (1440000,)
zarr_chunk_array: [[276 276 276 ... 203 203 203]
 [276 276 276 ... 203 203 203]
 [276 276 276 ... 203 203 203]
 ...
 [ 40  40  40 ...  40  40  40]
 [ 40  40  40 ...  40  40  40]
 [ 40  40  40 ...  40  40  40]]
min: 40.0
mean: 156.764391
max: 276.0
count: 16000000


In [13]:
# # Get filename patterns from the GeoTIFF URIs if you rely on them later
# gross_emis_CO2_folder_interval = gross_emis_CO2_folder.replace("INTERVAL", interval_label)
# gross_emis_CO2_uris = list_folder_uris(gross_emis_CO2_folder_interval)
# gross_emis_non_CO2_folder_interval = gross_emis_non_CO2_folder.replace("INTERVAL", interval_label)
# gross_emis_non_CO2_uris = list_folder_uris(gross_emis_non_CO2_folder_interval)
# gross_emis_all_gases_folder_interval = gross_emis_all_gases_folder.replace("INTERVAL", interval_label)
# gross_emis_all_gases_uris = list_folder_uris(gross_emis_all_gases_folder_interval)
# gross_remv_all_pools_folder_interval = gross_remv_all_pools_folder.replace("INTERVAL", interval_label)
# gross_remv_all_pools_uris = list_folder_uris(gross_remv_all_pools_folder_interval)
# net_flux_all_pools_CO2_folder_interval = net_flux_all_pools_CO2_folder.replace("INTERVAL", interval_label)
# net_flux_all_pools_CO2_uris = list_folder_uris(net_flux_all_pools_CO2_folder_interval)
# net_flux_all_pools_all_gases_folder_interval = net_flux_all_pools_all_gases_folder.replace("INTERVAL", interval_label)
# net_flux_all_pools_all_gases_uris = list_folder_uris(net_flux_all_pools_all_gases_folder_interval)
# node_folder_interval = node_folder.replace("INTERVAL", interval_label)
# node_tile_year_uris = list_folder_uris(node_folder_interval)
# 
# gross_emis_CO2_output_pattern       = parse_pattern_from_uri(gross_emis_CO2_uris)
# gross_emis_non_CO2_output_pattern   = parse_pattern_from_uri(gross_emis_non_CO2_uris)
# gross_emis_all_gases_output_pattern = parse_pattern_from_uri(gross_emis_all_gases_uris)
# gross_remv_all_pools_output_pattern = parse_pattern_from_uri(gross_remv_all_pools_uris)
# net_flux_CO2_output_pattern         = parse_pattern_from_uri(net_flux_all_pools_CO2_uris)
# net_flux_all_gases_output_pattern   = parse_pattern_from_uri(net_flux_all_pools_all_gases_uris)
# node_output_pattern                 = 'land_state_node'

gross_emis_CO2_output_pattern       = "gross_emissions__all_C_pools__CO2_only__MgCO2_ha_yr"
gross_emis_non_CO2_output_pattern   = "gross_emissions__all_C_pools__non_CO2_only__MgCO2e_ha_yr"
gross_emis_all_gases_output_pattern = "gross_emissions__all_C_pools__all_gases__MgCO2e_ha_yr"
gross_remv_all_pools_output_pattern = "gross_removals__all_C_pools__MgCO2_ha_yr"
net_flux_CO2_output_pattern         = "net_flux__all_C_pools__CO2_only__MgCO2_ha_yr"
net_flux_all_gases_output_pattern   = "net_flux__all_C_pools__all_gases__MgCO2e_ha_yr"
C_stock_non_soil_output_pattern     = "carbon_density__non_soil__MgC_ha"
node_output_pattern                 = "land_state_node"

# print(gross_emis_CO2_output_pattern)
# print(gross_emis_non_CO2_output_pattern)
# print(gross_emis_all_gases_output_pattern)
# print(gross_remv_all_pools_output_pattern)
# print(net_flux_CO2_output_pattern)
# print(gross_emis_CO2_output_pattern)
# print(net_flux_all_gases_output_pattern)
# print(C_stock_non_soil_output_pattern)
# print(node_output_pattern)

In [14]:
# Opens flux model output mega-zarr
ds = xr.open_zarr(veg_mega_zarr_s3_path, consolidated=False)
ds

<xarray.Dataset> Size: 998TB
Dimensions:                                                   (year: 9,
                                                               y: 720000,
                                                               x: 1440000)
Coordinates:
  * x                                                         (x) float64 12MB ...
  * year                                                      (year) int64 72B ...
  * y                                                         (y) float64 6MB ...
Data variables: (12/29)
    carbon_density__deadwood_C__MgC_ha                        (year, y, x) float32 37TB dask.array<chunksize=(9, 4000, 4000), meta=np.ndarray>
    carbon_density__BGC__MgC_ha                               (year, y, x) float32 37TB dask.array<chunksize=(9, 4000, 4000), meta=np.ndarray>
    carbon_density__AGC__MgC_ha                               (year, y, x) float32 37TB dask.array<chunksize=(9, 4000, 4000), meta=np.ndarray>
    carbon_density__litter_C__MgC_ha                          (year, y, x) float32 37TB dask.array<chunksize=(9, 4000, 4000), meta=np.ndarray>
    gross_emissions__N2O__MgCO2e_ha_yr                        (year, y, x) float32 37TB dask.array<chunksize=(9, 4000, 4000), meta=np.ndarray>
    composite_primary_forest                                  (year, y, x) uint8 9TB dask.array<chunksize=(9, 4000, 4000), meta=np.ndarray>
    ...                                                        ...
    net_flux__BGC__MgCO2_ha_yr                                (year, y, x) float32 37TB dask.array<chunksize=(9, 4000, 4000), meta=np.ndarray>
    land_state_node                                           (year, y, x) uint32 37TB dask.array<chunksize=(9, 4000, 4000), meta=np.ndarray>
    net_flux__AGC__MgCO2_ha_yr                                (year, y, x) float32 37TB dask.array<chunksize=(9, 4000, 4000), meta=np.ndarray>
    gross_removals__litter_C__MgCO2_ha_yr                     (year, y, x) float32 37TB dask.array<chunksize=(9, 4000, 4000), meta=np.ndarray>
    spatial_ref                                               int32 4B ...
    net_flux__deadwood_C__MgCO2_ha_yr                         (year, y, x) float32 37TB dask.array<chunksize=(9, 4000, 4000), meta=np.ndarray>

In [15]:
ds.chunksizes

Frozen({'year': (9,), 'y': (4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 

In [16]:
# Opens non-model output zarrs
adm0_xr = xr.open_zarr(adm0_zarr_path, consolidated=False).rename_vars(band_data='adm0')
pixel_area_xr = xr.open_zarr(pixel_area_zarr_path, consolidated=False).rename_vars(band_data='pixel_area')
primary_forest_IFL_xr = xr.open_zarr(primary_forest_IFL_zarr_path, consolidated=False).rename_vars(band_data='primary_forest_IFL')
WDPA_xr = xr.open_zarr(wdpa_zarr_path, consolidated=False).rename_vars(band_data='WDPA')
BRA_biomes_xr = xr.open_zarr(BRA_biomes_zarr_path, consolidated=False).rename_vars(band_data='BRA_biomes')
cont_eco_xr = xr.open_zarr(cont_eco_zarr_path, consolidated=False).rename_vars(band_data='cont_eco')
pixel_area_xr
# print(primary_forest_IFL_xr)

<xarray.Dataset> Size: 6TB
Dimensions:      (x: 1440000, y: 560000)
Coordinates:
  * x            (x) float64 12MB -180.0 -180.0 -180.0 ... 180.0 180.0 180.0
  * y            (y) float64 4MB 80.0 80.0 80.0 80.0 ... -60.0 -60.0 -60.0 -60.0
    band         int64 8B ...
Data variables:
    spatial_ref  int64 8B ...
    pixel_area   (y, x) float64 6TB dask.array<chunksize=(10000, 10000), meta=np.ndarray>

In [17]:
# Creates dataframe of state_node codes and meanings
state_node_df = create_state_node_df(state_node_lookup_table_local, state_node_lookup_table_s3, sheet)
node_codes = np.array(list(state_node_df['state_nodes']), dtype=np.uint32)
# node_codes

In [69]:
%%time

#TODO Solomon says that analysis layers need to be recast to float64: "We found out that aggregating all the pixel values over large areas hits float32 limits which will give very wrong results."
selected_analysis_vars = [
                 gross_emis_CO2_output_pattern,
                 gross_emis_non_CO2_output_pattern,
                 gross_emis_all_gases_output_pattern,
                 gross_remv_all_pools_output_pattern,
                 net_flux_CO2_output_pattern,
                 net_flux_all_gases_output_pattern,
                 C_stock_non_soil_output_pattern
]

# Define bounding box
# west, south, east, north = -180, -70, 180, 80   # Global
west, south, east, north = 21, -7, 22, -6  # Test chunk (00N_020E)
# west, south, east, north = 8, -13, 35, 13    # Minimal Central Africa bounding box (27x26 deg)
# west, south, east, north = 7, -15, 37, 15    # Slightly expanded Central Africa bounding box (30x30 deg)
# west, south, east, north = 0, -20, 40, 20    # Expanded Central Africa bounding box (40x40 deg)

ds_selected_analysis_vars = ds[selected_analysis_vars]
ds_selected_analysis_vars

print(f"Rounding coordinates: {timestr()}")
reference = round_coords(pixel_area_xr["pixel_area"])
ds_selected_analysis_vars = round_coords(ds_selected_analysis_vars)
adm0_xr = round_coords(adm0_xr)
primary_forest_IFL_xr = round_coords(primary_forest_IFL_xr)
WDPA_xr = round_coords(WDPA_xr)
BRA_biomes_xr = round_coords(BRA_biomes_xr)
land_state_node = round_coords(ds["land_state_node"])

print(f"Cropping: {timestr()}")
pixel_area_aligned                       = reference
adm0_aligned                             = safe_crop(adm0_xr, reference)
primary_forest_IFL_aligned               = safe_crop(primary_forest_IFL_xr, reference)
WDPA_aligned                             = safe_crop(WDPA_xr, reference)
BRA_biomes_aligned                       = safe_crop(BRA_biomes_xr, reference)
cont_eco_aligned                         = safe_crop(cont_eco_xr, reference)
land_state_node_aligned                  = safe_crop(land_state_node, reference)
ds_selected_analysis_vars_aligned        = safe_crop(ds_selected_analysis_vars, reference)

print(f"Creating flux cube: {timestr()}")
# List of selected variable names (already aligned and cropped)
selected_datasets = list(ds_selected_analysis_vars_aligned.data_vars)

# Expand pixel_area to match shape of flux variables
pixel_area_expanded = pixel_area_aligned.expand_dims(year=ds_selected_analysis_vars_aligned.year)

# Use the exact same x/y coordinates for both
x_coords = reference.coords['x']
y_coords = reference.coords['y']

# Replace coords in both sources
# pixel_area_expanded = pixel_area_expanded.assign_coords(x=x_coords, y=y_coords)
ds_selected_analysis_vars_aligned = ds_selected_analysis_vars_aligned.assign_coords(x=x_coords, y=y_coords)

# Multiply each flux var by pixel_area
flux_layers = []
for var in selected_datasets:
    flux_scaled = ((ds_selected_analysis_vars_aligned[var] * pixel_area_expanded) / 10000).astype("float32")
    flux_layers.append(flux_scaled)

# Convert pixel_area from m² to hectares, then adds to the list of layers to analyze (to get area of contextual layers)
pixel_area_layer = (pixel_area_expanded / 10000).astype("float32")
flux_layers.append(pixel_area_layer)

# Also updates the list of analysis layer names
selected_datasets.append("pixel_area_ha")

# Stack into one flux cube: shape (analysis_layer, year, y, x)
flux_cube = xr.concat(flux_layers, dim="analysis_layer")

# Set the analysis_layer coordinate names
flux_cube = flux_cube.assign_coords(
    analysis_layer=("analysis_layer", selected_datasets)
)
flux_cube = round_coords(flux_cube)

# Subset the flux cube by x/y coordinates
flux_cube_subset = flux_cube.sel(
    x=slice(west, east),
    y=slice(north, south)  # Note: y typically decreases from top to bottom
)
adm0_aligned_subset = adm0_aligned.sel(x=slice(west, east), y=slice(north, south))
primary_forest_IFL_aligned_subset = primary_forest_IFL_aligned.sel(x=slice(west, east), y=slice(north, south))
WDPA_aligned_subset = WDPA_aligned.sel(x=slice(west, east), y=slice(north, south))
BRA_biomes_aligned_subset = BRA_biomes_aligned.sel(x=slice(west, east), y=slice(north, south))
cont_eco_aligned_subset = cont_eco_aligned.sel(x=slice(west, east), y=slice(north, south))
land_state_node_aligned_subset = land_state_node_aligned.sel(x=slice(west, east), y=slice(north, south))
pixel_area_expanded_subset = pixel_area_expanded.sel(x=slice(west, east), y=slice(north, south))

print("Flux cube x range:", flux_cube_subset.coords['x'].values.min(), flux_cube_subset.coords['x'].values.max(), "len:", len(flux_cube_subset.coords['x']))
print("Pixel area x range:", pixel_area_expanded_subset.coords['x'].values.min(), pixel_area_expanded_subset.coords['x'].values.max(), "len:", len(pixel_area_expanded_subset.coords['x']))
print("land_state_node x range:", land_state_node_aligned_subset.coords['x'].values.min(), land_state_node_aligned_subset.coords['x'].values.max(), "len:", len(land_state_node_aligned_subset.coords['x']))

# For datasets that don't have global coverage
try:
    print("ADM0 x range:", adm0_aligned_subset["adm0"].coords['x'].values.min(), adm0_aligned_subset["adm0"].coords['x'].values.max(), "len:", len(adm0_aligned_subset["adm0"].coords['x']))
except:
    print("  ADM0 not in chunk extent")

try:
    print("IFL x range:", primary_forest_IFL_aligned_subset["primary_forest_IFL"].coords['x'].values.min(), primary_forest_IFL_aligned_subset["primary_forest_IFL"].coords['x'].values.max(), "len:", len(primary_forest_IFL_aligned_subset["primary_forest_IFL"].coords['x']))
except:
    print("  IFL/primary forest not in chunk extent")

try:
    print("WDPA x range:", WDPA_aligned_subset["WDPA"].coords['x'].values.min(), WDPA_aligned_subset["WDPA"].coords['x'].values.max(), "len:", len(WDPA_aligned_subset["WDPA"].coords['x']))
except:
    print("  WDPA not in chunk extent")

try:
    print("BRA biomes x range:", BRA_biomes_aligned_subset["BRA_biomes"].coords['x'].values.min(), BRA_biomes_aligned_subset["BRA_biomes"].coords['x'].values.max(), "len:", len(BRA_biomes_aligned_subset["BRA_biomes"].coords['x']))
except:
    print("  BRA biomes not in chunk extent")

try:
    print("cont_eco x range:", cont_eco_aligned_subset["cont_eco"].coords['x'].values.min(), cont_eco_aligned_subset["cont_eco"].coords['x'].values.max(), "len:", len(cont_eco_aligned_subset["cont_eco"].coords['x']))
except:
    print("  cont_eco not in chunk extent")

# Final alignment 
print(f"Aligning: {timestr()}")
(flux_cube_subset, 
pixel_area_expanded_subset, 
adm0_aligned_subset, 
primary_forest_IFL_aligned_subset, 
WDPA_aligned_subset,
cont_eco_aligned_subset,
land_state_node_aligned_subset,
) = xr.align(
    flux_cube_subset, 
    pixel_area_expanded_subset, 
    adm0_aligned_subset["adm0"], 
    primary_forest_IFL_aligned_subset["primary_forest_IFL"], 
    WDPA_aligned_subset["WDPA"], 
    cont_eco_aligned_subset["cont_eco"],
    land_state_node_aligned_subset, 
    join="override"
)

flux_cube_subset = flux_cube_subset.persist()

Rounding coordinates: 20260206_15_53_55
Cropping: 20260206_15_53_55
Creating flux cube: 20260206_15_54_05
Flux cube x range: 21.00012 21.99988 len: 4000
Pixel area x range: 21.00012 21.99988 len: 4000
land_state_node x range: 21.00012 21.99987 len: 4000
ADM0 x range: 21.00012 21.99988 len: 4000
IFL x range: 21.00012 21.99988 len: 4000
WDPA x range: 21.00012 21.99988 len: 4000
  BRA biomes not in chunk extent
cont_eco x range: 21.000125 21.999875 len: 4000
Aligning: 20260206_15_54_23
CPU times: user 26.7 s, sys: 993 ms, total: 27.7 s
Wall time: 29.3 s


In [67]:
%%time

print(f"Computing: {timestr()}")
results = xarray_reduce(
    flux_cube_subset,
    *(
      adm0_aligned_subset, 
      land_state_node_aligned_subset, 
      primary_forest_IFL_aligned_subset, 
      WDPA_aligned_subset, 
      cont_eco_aligned_subset,
      flux_cube_subset["year"]
    ),
    func='sum',
    expected_groups=(
        gadm_adm0_ids, 
        node_codes, 
        primary_forest_IFL_codes, 
        WDPA_codes, 
        cont_eco_codes,
        flux_cube_subset.year.values,
    ),
    group_dims=["year"],
    reindex=ReindexStrategy(blockwise=False, array_type=ReindexArrayType.SPARSE_COO),
    fill_value=0
).compute()

# Contextual layers to use to merge pixel_area against other analysis layers (to calculate flux/ha)
contextual_layers = [
    'adm0', 
    'land_state_node', 
    'year', 
    'primary_forest_IFL',
    'cont_eco',
    'WDPA'
]

coord_dict = convert_to_coord_dict(results)
# coord_dict
df = create_df(coord_dict, state_node_df, contextual_layers)
print(f"Done: {timestr()}")
print(f"Rows in dataframe: {len(df.index)}")
df.head()

Computing: 20260206_15_52_56
   Postprocessing: 20260206_15_53_09
Done: 20260206_15_53_09
Rows in dataframe: 2604
CPU times: user 295 ms, sys: 11.2 ms, total: 306 ms
Wall time: 13.2 s


,analysis_layer,adm0,land_state_node,primary_forest_IFL,WDPA,cont_eco,year,value,pixel_area_ha,meaning,broad_class,detailed_class,country_name,region,flux_Mg_ha
0,gross_emissions__all_C_pools__CO2_only__MgCO2_...,COD,31221200,0,0,1020,2018,94.842934,0.306061,Full loss of non-oil palm tree crops as short ...,tree,tree_loss,Democratic Republic of the Congo,Tropical Africa,309.882141
1,gross_emissions__all_C_pools__CO2_only__MgCO2_...,COD,31221200,0,0,1020,2019,149.264893,0.306061,Full loss of non-oil palm tree crops as short ...,tree,tree_loss,Democratic Republic of the Congo,Tropical Africa,487.696503
2,gross_emissions__all_C_pools__CO2_only__MgCO2_...,COD,31221200,0,0,1020,2020,299.738739,0.918182,Full loss of non-oil palm tree crops as short ...,tree,tree_loss,Democratic Republic of the Congo,Tropical Africa,326.447906
3,gross_emissions__all_C_pools__CO2_only__MgCO2_...,COD,31221200,0,0,1020,2021,34.112473,0.153031,Full loss of non-oil palm tree crops as short ...,tree,tree_loss,Democratic Republic of the Congo,Tropical Africa,222.912750
4,gross_emissions__all_C_pools__CO2_only__MgCO2_...,COD,31221200,0,0,1020,2024,4.017562,0.076515,Full loss of non-oil palm tree crops as short ...,tree,tree_loss,Democratic Republic of the Congo,Tropical Africa,52.506668


In [65]:
print(df[(df.analysis_layer == 'gross_emissions__all_C_pools__CO2_only__MgCO2_ha_yr') & (df.year == 2016)]['value'].sum().round())
print(df[(df.analysis_layer == 'gross_emissions__all_C_pools__CO2_only__MgCO2_ha_yr') & (df.year == 2017)]['value'].sum().round())
print(df[(df.analysis_layer == 'gross_emissions__all_C_pools__CO2_only__MgCO2_ha_yr') & (df.year == 2018)]['value'].sum().round())
print(df[(df.analysis_layer == 'net_flux__all_C_pools__all_gases__MgCO2e_ha_yr') & (df.year == 2023)]['value'].sum().round())
print(df[(df.analysis_layer == 'net_flux__all_C_pools__all_gases__MgCO2e_ha_yr') & (df.year == 2024)]['value'].sum().round())

10498938.0
11048152.0
9166699.0
6988055.0
5668196.0


In [ ]:
# Export to a csv so data can be used in Excel or reused
df.to_csv('/mnt/c/GIS/vegetation_flux_zonal_stats_AUS_v_1_0_3__20251211.csv', index=False)

In [68]:
pivot_fields=contextual_layers+['meaning', 'broad_class', 'detailed_class']
df_wide = df.pivot(index = pivot_fields, columns="analysis_layer", values="value").reset_index()
# df_wide = df.pivot(index=['land_state_node', 'year', 'primary_forest_IFL', 'adm0'], columns="analysis_layer", values="value").reset_index()
df_wide

analysis_layer,adm0,land_state_node,year,primary_forest_IFL,cont_eco,WDPA,meaning,broad_class,detailed_class,gross_emissions__all_C_pools__CO2_only__MgCO2_ha_yr,gross_emissions__all_C_pools__all_gases__MgCO2e_ha_yr,gross_emissions__all_C_pools__non_CO2_only__MgCO2e_ha_yr,gross_removals__all_C_pools__MgCO2_ha_yr,net_flux__all_C_pools__CO2_only__MgCO2_ha_yr,net_flux__all_C_pools__all_gases__MgCO2e_ha_yr
0,COD,21200000,2020,0,1020,0,Gain of non-oil palm planted trees,tree,tree_gain,NaN,NaN,NaN,-4.017565,-4.017565,-4.017565
1,COD,21200000,2021,0,1020,0,Gain of non-oil palm planted trees,tree,tree_gain,NaN,NaN,NaN,-9.039507,-9.039507,-9.039507
2,COD,21200000,2022,0,1020,0,Gain of non-oil palm planted trees,tree,tree_gain,NaN,NaN,NaN,-6.026340,-6.026340,-6.026340
3,COD,21200000,2023,0,1020,0,Gain of non-oil palm planted trees,tree,tree_gain,NaN,NaN,NaN,-4.017560,-4.017560,-4.017560
4,COD,22100000,2016,0,1020,0,Gain of terrestrial natural forest,tree,tree_gain,NaN,NaN,NaN,-26016.833984,-26016.833984,-26016.833984
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
598,COD,63900000,2022,1,1020,0,Short vegetation remaining short vegetation wi...,short_veg,short_veg_short_veg_undisturbed,NaN,1332.592407,1332.592407,NaN,NaN,1332.592407
599,COD,63900000,2023,0,1020,0,Short vegetation remaining short vegetation wi...,short_veg,short_veg_short_veg_undisturbed,NaN,7418.605957,7418.605957,NaN,NaN,7418.605957
600,COD,63900000,2023,1,1020,0,Short vegetation remaining short vegetation wi...,short_veg,short_veg_short_veg_undisturbed,NaN,1035.995728,1035.995728,NaN,NaN,1035.995728
601,COD,63900000,2024,0,1020,0,Short vegetation remaining short vegetation wi...,short_veg,short_veg_short_veg_undisturbed,NaN,9970.387695,9970.387695,NaN,NaN,9970.387695


In [ ]:
walker = pyg.walk(df_wide)

In [ ]:
try:
    client.shutdown()
except Exception:
    pass